# 04 - Advanced Retail Analytics: Enrichment Impact & ML Dataset Preparation

Notebook ini bertujuan untuk menguji dampak dari *enrichment data* (promosi, cuaca, dan risiko inventaris) terhadap performa bisnis, serta membangun *feature table* tingkat harian (`daily_demand`) sebagai fondasi dataset *Machine Learning Demand Forecasting*.

**Ruang Lingkup Analysis:**
1. **Promotion Impact:** Mengukur efektivitas jenis promosi terhadap total transaksi, kuantitas penjualan, dan revenue.
2. **Weather Impact:** Menganalisis pengaruh kondisi cuaca terhadap perilaku pembelian di setiap lokasi toko.
3. **Inventory Risk:** Mengidentifikasi tingkat persediaan, produk yang sering habis (*stockout*), dan risiko persediaan.
4. **Daily Demand Dataset Construction:** Membentuk dataset dasar agregasi daily demand (Tanggal x Toko x Produk).
5. **Feature Enrichment (Calendar & Weather):** Menggabungkan data kalender dan cuaca berbasis kunci kombinasi presisi `(transaction_date, store_location)` untuk menghindari *row duplication*.

## 1. Setup Environment & Database Connection

Menginisialisasi path direktori proyek serta membuat koneksi ke PostgreSQL database `smart_retail`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Set path root project untuk mengimpor modul custom
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from etl.db_connection import get_engine

# Inisialisasi koneksi engine PostgreSQL
engine = get_engine()

## 2. Promotion Impact Analysis

Mengevaluasi kontribusi setiap jenis promosi (`promotion_type`) terhadap jumlah transaksi, unit barang terjual, dan total pendapatan (*revenue*).

In [2]:
# Query SQL untuk mengukur efektivitas promosi berdasarkan rentang tanggal aktif promosi
query_promotion = """
SELECT
    p.promotion_type,
    COUNT(DISTINCT p.promotion_id) AS promotions,
    COUNT(DISTINCT s.transaction_id) AS transactions,
    SUM(s.transaction_qty) AS quantity,
    ROUND(SUM(s.transaction_qty * s.unit_price), 2) AS revenue
FROM promotions p
LEFT JOIN sales_transactions s
    ON s.store_id = p.store_id
    AND s.product_id = p.product_id
    AND s.transaction_date
        BETWEEN p.start_date AND p.end_date
GROUP BY p.promotion_type
ORDER BY revenue DESC;
"""

# Read data dari database ke pandas DataFrame
promotion_impact = pd.read_sql(query_promotion, engine)
promotion_impact

,promotion_type,promotions,transactions,quantity,revenue
0,Weekend Promotion,35,856,1217,3976.30
1,Seasonal Promotion,9,186,253,775.85
2,Product Discount,6,101,158,431.15
3,Happy Hour,4,42,51,220.20


## 3. Weather Impact Analysis

Menganalisis dampak kondisi cuaca (`Clear`, `Cloudy`, `Rain`) terhadap volume transaksi dan total pendapatan di lokasi toko yang sesuai.

In [3]:
# Query SQL untuk menggabungkan transaksi dengan cuaca berdasarkan tanggal dan lokasi toko
query_weather = """
SELECT
    w.weather_condition,
    COUNT(DISTINCT s.transaction_id) AS transactions,
    SUM(s.transaction_qty) AS quantity,
    ROUND(
        SUM(s.transaction_qty * s.unit_price),
        2
    ) AS revenue
FROM weather w
JOIN sales_transactions s
    ON s.transaction_date = w.date
JOIN stores st
    ON s.store_id = st.store_id
WHERE st.store_location = w.location
GROUP BY w.weather_condition
ORDER BY revenue DESC;
"""

# Read data dari database
weather_impact = pd.read_sql(query_weather, engine)
weather_impact

,weather_condition,transactions,quantity,revenue
0,Cloudy,68431,99116,321214.99
1,Clear,59528,84959,277469.57
2,Rain,21157,30395,100127.77


## 4. Inventory Risk Analysis

Mendeteksi potensi masalah persediaan seperti rata-rata stok akhir (*closing stock*) dan jumlah hari terjadinya stok habis (*stockout_days*) per produk dan lokasi toko.

In [4]:
# Query SQL untuk menganalisis pergerakan stok dan menghitung frekuensi kejadian stockout
query_inventory = """
SELECT
    st.store_location,
    p.product_detail,
    SUM(i.sold_quantity) AS total_sold,
    AVG(i.closing_stock) AS avg_closing_stock,
    MAX(i.closing_stock) AS max_closing_stock,
    SUM(
        CASE
            WHEN i.stockout_flag THEN 1
            ELSE 0
        END
    ) AS stockout_days
FROM inventory i
JOIN stores st
    ON i.store_id = st.store_id
JOIN products p
    ON i.product_id = p.product_id
GROUP BY
    st.store_location,
    p.product_id,
    p.product_detail
ORDER BY
    stockout_days DESC,
    total_sold DESC;
"""

# Read data dan tampilkan 20 baris pertama dengan risiko stockout tertinggi
inventory_risk = pd.read_sql(query_inventory, engine)
inventory_risk.head(20)

,store_location,product_detail,total_sold,avg_closing_stock,max_closing_stock,stockout_days
0,Lower Manhattan,Peppermint Lg,1582,35.364641,94,1
1,Astoria,Spicy Eye Opener Chai Rg,1520,34.872928,99,1
2,Hell's Kitchen,Ouro Brasileiro shot,1854,83.116022,147,0
3,Astoria,Dark chocolate Lg,1755,89.011050,141,0
4,Astoria,Earl Grey Rg,1725,94.248619,160,0
5,Astoria,Peppermint Rg,1673,59.723757,124,0
6,Astoria,Spicy Eye Opener Chai Lg,1634,74.453039,133,0
7,Astoria,Ethiopia Sm,1619,66.784530,119,0
8,Astoria,Columbian Medium Roast Rg,1613,74.397790,145,0
9,Hell's Kitchen,Serenity Green Tea Rg,1601,109.033149,168,0


## 5. Daily Demand Dataset Construction

Membangun dataset agregat tingkat harian (`transaction_date` x `store_id` x `product_id`) sebagai tabel fitur dasar (*base feature table*) untuk model Machine Learning Demand Forecasting.

In [5]:
# Query SQL agregasi demand harian per toko dan per produk
query_daily_demand = """
SELECT
    s.transaction_date,
    s.store_id,
    s.product_id,

    SUM(s.transaction_qty) AS demand,
    SUM(s.transaction_qty * s.unit_price) AS revenue,

    COUNT(DISTINCT s.transaction_id) AS transactions

FROM sales_transactions s

GROUP BY
    s.transaction_date,
    s.store_id,
    s.product_id

ORDER BY
    s.transaction_date,
    s.store_id,
    s.product_id;
"""

# Execute query dan konversi kolom tanggal ke format datetime
daily_demand = pd.read_sql(
    query_daily_demand,
    engine,
    parse_dates=["transaction_date"]
)

# Tampilkan jumlah baris dan 5 data pertama
print("Rows:", len(daily_demand))
daily_demand.head()

Rows: 31764


,transaction_date,store_id,product_id,demand,revenue,transactions
0,2023-01-01,3,22,5,10.0,5
1,2023-01-01,3,23,7,17.5,5
2,2023-01-01,3,24,5,15.0,4
3,2023-01-01,3,25,5,11.0,3
4,2023-01-01,3,26,4,12.0,2


## 6. Enriching Daily Demand with Calendar Feature

Menggabungkan fitur kalender/waktu (`calendar.csv`) seperti nama hari, minggu, dan penanda *weekend* ke dalam dataset `daily_demand`.

In [6]:
# Load dataset kalender dari CSV
calendar = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "calendar.csv",
    parse_dates=["date"]
)

# Merge dataset calendar berdasarkan tanggal transaksi
daily_demand = daily_demand.merge(
    calendar,
    left_on="transaction_date",
    right_on="date",
    how="left"
)

# Cek sampel data setelah penggabungan kalender
print("Rows after calendar merge:", len(daily_demand))
daily_demand.head(2)

Rows after calendar merge: 31764


,transaction_date,store_id,product_id,demand,revenue,transactions,date,year,month,month_name,day,day_of_week,day_name,week_of_year,is_weekend
0,2023-01-01,3,22,5,10.0,5,2023-01-01,2023,1,January,1,6,Sunday,52,True
1,2023-01-01,3,23,7,17.5,5,2023-01-01,2023,1,January,1,6,Sunday,52,True


## 7. Enriching Daily Demand with Store Location & Weather Data

Menggabungkan atribut lokasi toko dan data cuaca. **Catatan Penting:** Penggabungan data cuaca wajib menggunakan kombinasi kunci `(transaction_date, store_location)` agar tidak menghasilkan duplikasi *many-to-many join*.

In [7]:
# 1. Ambil data lokasi toko dari PostgreSQL
stores = pd.read_sql(
    """
    SELECT store_id, store_location
    FROM stores
    """,
    engine
)

# 2. Merge informasi lokasi toko ke daily_demand
daily_demand = daily_demand.merge(
    stores,
    on="store_id",
    how="left"
)

# 3. Load dataset cuaca dari CSV
weather = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "weather.csv",
    parse_dates=["date"]
)

# 4. Merge data cuaca dengan kombinasi kunci Tanggal + Lokasi Toko yang presisi
daily_demand = daily_demand.merge(
    weather,
    left_on=[
        "transaction_date",
        "store_location"
    ],
    right_on=[
        "date",
        "location"
    ],
    how="left"
)

# 5. Validasi hasil penggabungan akhir
print("Rows:", len(daily_demand))
print(
    daily_demand[
        [
            "transaction_date",
            "store_id",
            "store_location",
            "temperature",
            "rainfall_mm",
            "weather_condition"
        ]
    ].head()
)

Rows: 31764
  transaction_date  store_id store_location  temperature  rainfall_mm  \
0       2023-01-01         3        Astoria         5.91          0.0   
1       2023-01-01         3        Astoria         5.91          0.0   
2       2023-01-01         3        Astoria         5.91          0.0   
3       2023-01-01         3        Astoria         5.91          0.0   
4       2023-01-01         3        Astoria         5.91          0.0   

  weather_condition  
0             Clear  
1             Clear  
2             Clear  
3             Clear  
4             Clear  
